In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH350=extractor("Dati/Tprime_tAq_1800_MH350_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH350_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH350[mask]


In [ ]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_plot, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_plot) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_plot) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=250, sigma=5, gamma=1)
m_voigt.limits["mu"]= (225, 275)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


In [ ]:
fit_MH350_values={}
fit_MH350_errors={}

fit_values={'MH350': fit_MH350_values,}
fit_errors={'MH350_errors': fit_MH350_errors}

for param in m_voigt.parameters:
    fit_MH350_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH350_errors[error] = m_voigt.errors[error]

print(fit_MH350_values)
print(fit_MH350_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH350"]=fit_MH350_values
results["MH350_errors"]=fit_MH350_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH350"]=fit_MH350_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH350_errors"]=fit_MH350_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  
